# Phase 2, Step 6: Combine All Features

This notebook merges all engineered features into a single dataset for machine learning modeling. It aligns features by **location** (Latitude, Longitude) and **Sample Date** to ensure correct spatial-temporal correspondence.

## Objectives

1. Load all feature datasets (Landsat, TerraClimate, spatial, temporal)
2. Merge features on (Latitude, Longitude, Sample Date)
3. Add target variables for training
4. Save combined training and validation datasets ready for modeling

## Step 1: Load Dependencies and File Paths

In [3]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import os

In [4]:
# File paths - use enhanced/new versions if available, else fallback to base versions
LANDSAT_TRAIN = "landsat_features_training_enhanced.csv"
LANDSAT_VAL = "landsat_features_validation_enhanced.csv"
LANDSAT_TRAIN_ALT = "landsat_features_training.csv"
LANDSAT_VAL_ALT = "landsat_features_validation.csv"

# Extra Landsat bands (red, blue, lwir11 + derived indices from 04_Landsat_additiona-bands_feature.ipynb)
LANDSAT_EXTRA_TRAIN = "landsat_extra_bands_training.csv"
LANDSAT_EXTRA_VAL   = "landsat_extra_bands_validation.csv"

TERRACLIMATE_TRAIN = "terraclimate_features_training_new_variables.csv"
TERRACLIMATE_VAL = "terraclimate_features_validation_new_variables.csv"
TERRACLIMATE_TRAIN_ALT = "terraclimate_features_training.csv"
TERRACLIMATE_VAL_ALT = "terraclimate_features_validation.csv"

SPATIAL_TRAIN = "spatial_features_training.csv"
SPATIAL_VAL = "spatial_features_validation.csv"

TEMPORAL_TRAIN = "temporal_features_training.csv"
TEMPORAL_VAL = "temporal_features_validation.csv"

TRAINING_TARGETS = "water_quality_training_dataset.csv"
VALIDATION_TEMPLATE = "submission_template.csv"

MERGE_KEYS = ["Latitude", "Longitude", "Sample Date"]

## Step 2: Load All Feature Datasets

In [5]:
def load_csv(path, alt_path=None):
    """Load CSV, trying alt_path if primary not found."""
    if os.path.exists(path):
        return pd.read_csv(path)
    if alt_path and os.path.exists(alt_path):
        return pd.read_csv(alt_path)
    raise FileNotFoundError(f"Neither {path} nor {alt_path} found.")

# Landsat (enhanced or base)
landsat_train = load_csv(LANDSAT_TRAIN, LANDSAT_TRAIN_ALT)
landsat_val = load_csv(LANDSAT_VAL, LANDSAT_VAL_ALT)
print(f"Landsat training: {landsat_train.shape}")
print(f"Landsat validation: {landsat_val.shape}")

# Extra Landsat bands (red, blue, lwir11 + derived indices) — optional
try:
    landsat_extra_train = pd.read_csv(LANDSAT_EXTRA_TRAIN)
    landsat_extra_val   = pd.read_csv(LANDSAT_EXTRA_VAL)
    print(f"Landsat extra bands training: {landsat_extra_train.shape}")
    print(f"Landsat extra bands validation: {landsat_extra_val.shape}")
except FileNotFoundError:
    landsat_extra_train = landsat_extra_val = None
    print("Landsat extra bands not found — run 04_Landsat_additiona-bands_feature.ipynb first to include them.")

# TerraClimate
tc_train = load_csv(TERRACLIMATE_TRAIN, TERRACLIMATE_TRAIN_ALT)
tc_val = load_csv(TERRACLIMATE_VAL, TERRACLIMATE_VAL_ALT)
print(f"TerraClimate training: {tc_train.shape}")
print(f"TerraClimate validation: {tc_val.shape}")

# Spatial
spatial_train = pd.read_csv(SPATIAL_TRAIN)
spatial_val = pd.read_csv(SPATIAL_VAL)
print(f"Spatial training: {spatial_train.shape}")
print(f"Spatial validation: {spatial_val.shape}")

# Temporal
temporal_train = pd.read_csv(TEMPORAL_TRAIN)
temporal_val = pd.read_csv(TEMPORAL_VAL)
print(f"Temporal training: {temporal_train.shape}")
print(f"Temporal validation: {temporal_val.shape}")

# Training targets and validation template
training_targets = pd.read_csv(TRAINING_TARGETS)
val_template = pd.read_csv(VALIDATION_TEMPLATE)
print(f"Training targets: {training_targets.shape}")
print(f"Validation template: {val_template.shape}")

Landsat training: (9319, 27)
Landsat validation: (200, 27)
Landsat extra bands not found — run 04_Landsat_additiona-bands_feature.ipynb first to include them.
TerraClimate training: (9319, 9)
TerraClimate validation: (200, 9)
Spatial training: (9319, 5)
Spatial validation: (200, 5)
Temporal training: (9319, 15)
Temporal validation: (200, 15)
Training targets: (9319, 6)
Validation template: (200, 6)


## Step 3: Standardize Merge Keys and Normalize Sample Date

Sample dates may appear in different formats (e.g., DD-MM-YYYY vs YYYY-MM-DD). We ensure consistent parsing and string representation for reliable merging.

In [6]:
def normalize_date_and_coords(df, keys=MERGE_KEYS):
    """Parse Sample Date and round lat/lon for consistent merge keys."""
    out = df.copy()
    # Ensure consistent date format: DD-MM-YYYY string
    original_dates = out["Sample Date"].copy()
    dates = pd.to_datetime(original_dates, dayfirst=True, errors="coerce")
    parsed_str = dates.dt.strftime("%d-%m-%Y")
    # Where parsing failed (NaT), keep original to avoid NaN in merge keys
    out["Sample Date"] = parsed_str.where(dates.notna(), original_dates)
    # Small float tolerance: round to 6 decimals to avoid floating-point mismatch
    out["Latitude"] = out["Latitude"].round(6)
    out["Longitude"] = out["Longitude"].round(6)
    return out

def get_feature_cols(df, keys=MERGE_KEYS):
    return [c for c in df.columns if c not in keys]

# Normalize all dataframes
landsat_train = normalize_date_and_coords(landsat_train)
landsat_val = normalize_date_and_coords(landsat_val)
if landsat_extra_train is not None:
    landsat_extra_train = normalize_date_and_coords(landsat_extra_train)
    landsat_extra_val   = normalize_date_and_coords(landsat_extra_val)
tc_train = normalize_date_and_coords(tc_train)
tc_val = normalize_date_and_coords(tc_val)
spatial_train = normalize_date_and_coords(spatial_train)
spatial_val = normalize_date_and_coords(spatial_val)
temporal_train = normalize_date_and_coords(temporal_train)
temporal_val = normalize_date_and_coords(temporal_val)
training_targets = normalize_date_and_coords(training_targets)
val_template = normalize_date_and_coords(val_template)

print("Merge keys normalized (Sample Date as DD-MM-YYYY, lat/lon rounded to 6 decimals).")

Merge keys normalized (Sample Date as DD-MM-YYYY, lat/lon rounded to 6 decimals).


## Step 4: Merge All Features

We merge on **(Latitude, Longitude, Sample Date)**. The base for training is the water quality dataset (includes targets); the base for validation is the submission template.

In [7]:
def merge_features(base, *feature_dfs, keys=MERGE_KEYS):
    """Merge base with feature DataFrames on keys. Add only feature columns (no duplicate keys)."""
    result = base.copy()
    for df in feature_dfs:
        feature_cols = get_feature_cols(df, keys)
        if not feature_cols:
            continue
        result = result.merge(df[keys + feature_cols], on=keys, how="left")
    return result

# Build optional extra-bands lists (empty if file not found)
train_extra_list = [landsat_extra_train] if landsat_extra_train is not None else []
val_extra_list   = [landsat_extra_val]   if landsat_extra_val   is not None else []

# Training: base = water quality (has targets)
train_combined = merge_features(
    training_targets,
    landsat_train,
    *train_extra_list,
    tc_train,
    spatial_train,
    temporal_train,
)

# Validation: base = submission template
val_combined = merge_features(
    val_template,
    landsat_val,
    *val_extra_list,
    tc_val,
    spatial_val,
    temporal_val,
)

print(f"Combined training: {train_combined.shape}")
print(f"Combined validation: {val_combined.shape}")

Combined training: (9319, 50)
Combined validation: (200, 50)


## Step 5: Verify Alignment and Handle Missing Values

In [8]:
# Assert row counts match
assert len(train_combined) == len(training_targets), (
    f"Training row mismatch: combined {len(train_combined)} vs targets {len(training_targets)}"
)
assert len(val_combined) == len(val_template), (
    f"Validation row mismatch: combined {len(val_combined)} vs template {len(val_template)}"
)

# Missing value summary
train_missing = train_combined.isna().sum()
cols_with_missing = train_missing[train_missing > 0].sort_values(ascending=False)
if len(cols_with_missing) > 0:
    print("Columns with missing values (training):")
    print(cols_with_missing.head(20))
else:
    print("No missing values in training data.")

val_missing = val_combined.isna().sum()
val_cols_missing = val_missing[val_missing > 0].sort_values(ascending=False)
if len(val_cols_missing) > 0:
    print("\nColumns with missing values (validation):")
    print(val_cols_missing.head(20))
else:
    print("\nNo missing values in validation data.")

Columns with missing values (training):
nir                   1085
green                 1085
swir16                1085
swir22                1085
NDMI                  1085
MNDWI                 1085
NDVI                  1085
NDWI                  1085
NDSI_water            1085
NDTI                  1085
Turbidity_Index       1085
Chlorophyll_Proxy     1085
BSI                   1085
SWIR22_NIR_ratio      1085
SWIR16_NIR_ratio      1085
Green_NIR_ratio       1085
SWIR22_Green_ratio    1085
SWIR16_Green_ratio    1085
log_nir               1085
log_green             1085
dtype: int64

Columns with missing values (validation):
Total Alkalinity                 200
Electrical Conductance           200
Dissolved Reactive Phosphorus    200
nir                               19
green                             19
swir16                            19
swir22                            19
NDMI                              19
MNDWI                             19
NDVI                           

## Step 6: Save Combined Datasets

In [9]:
OUT_TRAIN = "combined_features_training.csv"
OUT_VAL = "combined_features_validation.csv"

train_combined.to_csv(OUT_TRAIN, index=False)
val_combined.to_csv(OUT_VAL, index=False)

print(f"Saved {OUT_TRAIN} ({len(train_combined)} rows, {len(train_combined.columns)} columns)")
print(f"Saved {OUT_VAL} ({len(val_combined)} rows, {len(val_combined.columns)} columns)")
print("\nColumn groups:")
print("  Merge keys:", MERGE_KEYS)
print("  Targets (training): Total Alkalinity, Electrical Conductance, Dissolved Reactive Phosphorus")
print("  Landsat, TerraClimate, Spatial, Temporal features:", len(get_feature_cols(landsat_train)) + len(get_feature_cols(tc_train)) + len(get_feature_cols(spatial_train)) + len(get_feature_cols(temporal_train)))

Saved combined_features_training.csv (9319 rows, 50 columns)
Saved combined_features_validation.csv (200 rows, 50 columns)

Column groups:
  Merge keys: ['Latitude', 'Longitude', 'Sample Date']
  Targets (training): Total Alkalinity, Electrical Conductance, Dissolved Reactive Phosphorus
  Landsat, TerraClimate, Spatial, Temporal features: 44


In [10]:
print("\nAll columns:")
for i, col in enumerate(train_combined.columns, 1):
    print(f"  {i:2d}. {col}")

print("\nFirst 5 rows (training):")
train_combined.head()


All columns:
   1. Latitude
   2. Longitude
   3. Sample Date
   4. Total Alkalinity
   5. Electrical Conductance
   6. Dissolved Reactive Phosphorus
   7. nir
   8. green
   9. swir16
  10. swir22
  11. NDMI
  12. MNDWI
  13. NDVI
  14. NDWI
  15. NDSI_water
  16. NDTI
  17. Turbidity_Index
  18. Chlorophyll_Proxy
  19. BSI
  20. SWIR22_NIR_ratio
  21. SWIR16_NIR_ratio
  22. Green_NIR_ratio
  23. SWIR22_Green_ratio
  24. SWIR16_Green_ratio
  25. log_nir
  26. log_green
  27. log_swir16
  28. log_swir22
  29. nir_squared
  30. swir22_squared
  31. pet
  32. ppt
  33. tmax
  34. def
  35. pdsi
  36. q
  37. elevation
  38. land_cover
  39. year
  40. month
  41. quarter
  42. day_of_year
  43. week_of_year
  44. season
  45. is_wet_season
  46. months_since_start
  47. month_sin
  48. month_cos
  49. day_of_year_sin
  50. day_of_year_cos

First 5 rows (training):


,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,nir,green,swir16,swir22,...,quarter,day_of_year,week_of_year,season,is_wet_season,months_since_start,month_sin,month_cos,day_of_year_sin,day_of_year_cos
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0,11190.0,11426.0,7687.5,7645.0,...,1,2,52,summer,1,0,0.5,0.866025,0.034328,0.999411
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0,17658.5,9550.0,13746.5,10574.0,...,1,3,1,summer,1,0,0.5,0.866025,0.051479,0.998674
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0,15210.0,10720.0,17974.0,14201.0,...,1,3,1,summer,1,0,0.5,0.866025,0.051479,0.998674
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0,14887.0,10943.0,13522.0,11403.0,...,1,3,1,summer,1,0,0.5,0.866025,0.051479,0.998674
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0,16828.5,9502.5,12665.5,9643.0,...,1,3,1,summer,1,0,0.5,0.866025,0.051479,0.998674


## Summary

The combined datasets are ready for machine learning modeling:

- **`combined_features_training.csv`**: Training data with targets (Total Alkalinity, Electrical Conductance, Dissolved Reactive Phosphorus) and all engineered features.
- **`combined_features_validation.csv`**: Validation data with same features; targets are empty for prediction.

Features are aligned by (Latitude, Longitude, Sample Date). You can proceed to model training and hyperparameter tuning.

### Feature sources included

| Source | File | Features |
|--------|------|----------|
| Landsat (base + enhanced) | `landsat_features_training_enhanced.csv` | nir, green, swir16, swir22, NDMI, MNDWI + 18 derived indices |
| Landsat extra bands | `landsat_extra_bands_training.csv` | red, blue, lwir11 + NDVI_red, EVI, SAVI, BSI_full, AWEI_nsh, AWEI_sh, NRI, Red_Blue_ratio, Green_Red_ratio, Turbidity_Red |
| TerraClimate | `terraclimate_features_training_new_variables.csv` | Climate variables |
| Spatial | `spatial_features_training.csv` | elevation, land_cover |
| Temporal | `temporal_features_training.csv` | year, month, season, cyclical encodings, etc. |

> If `landsat_extra_bands_training.csv` is missing, run `04_Landsat_additiona-bands_feature.ipynb` first. The notebook will still run without it.